# 🏦 GigScore — Agent 4: Risk Explanation + Final GigScore Calculator
## Behavioral Credit Intelligence for India's Informal Economy

**Barclays Hack-O-Hire 2026 · PS4: AI-Powered Alternate Credit Scoring**  
**Team GigScore · Plaksha University · March 28–29, 2026**

---

### What This Agent Does

Agent 4 is the **explanation and final scoring layer**. It sits between the ML model (Agent 2), the Social Trust Graph (Agent 3), and the output agents (Agent 5 credit story, Agent 6 loan structuring).

| Input | Source |
|-------|--------|
| `default_prob`, `raw_gigscore`, `shap_top3` | Agent 2 |
| `social_trust_score`, graph metrics | Agent 3 |
| All behavioral features | agent0_features.csv |

| Output | Destination |
|--------|-------------|
| `final_gigscore` (3-layer formula) | Agent 5, Agent 6 |
| `positive_signals` (top 3) | Agent 5 user narrative |
| `risk_factors` (top 3) | Agent 5 Barclays dashboard |
| `score_breakdown` | Agent 6 loan structuring |
| `reasoning_dict` | Agent 5, Agent 7 audit |

### The 3-Layer GigScore Formula

```
Layer 1 (90%): XGBoost base score — sigmoid inversion of default_prob
Layer 2 (10%): Social Trust Graph overlay — payment network stability  
Layer 3 (±):   Behavioral bonus/penalty — utility streak, net saving, obligation rate
               × Income Stability Multiplier — PLFS benchmarking
```

> *"No single layer can be gamed. All three have to align."*


In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg,
                    '--break-system-packages', '-q'], capture_output=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA', 'axes.facecolor': '#FFFFFF',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans', 'font.size': 11,
})

BARCLAYS_BLUE  = '#00AEEF'
BARCLAYS_NAVY  = '#1F4788'
BARCLAYS_EAGLE = '#D4002A'
GREEN          = '#2ECC71'
ORANGE         = '#F39C12'

print("✅ Libraries loaded.")


## Step 1 — Load Agent 2 + Agent 3 + Agent 0 Outputs

In [ ]:
# ── Load Agent 2 outputs ─────────────────────────────────────────────────
predictions_path = Path("GigScore_Predictions.csv")
shap_top3_path   = Path("GigScore_SHAP_Top3.csv")
features_path    = Path("agent0_features.csv")
graph_path       = Path("agent3_graph_scores.json")

if not predictions_path.exists():
    raise FileNotFoundError("GigScore_Predictions.csv not found. Run Agent 2 first.")
if not features_path.exists():
    raise FileNotFoundError("agent0_features.csv not found. Run Agent 0 first.")

predictions = pd.read_csv(predictions_path)
shap_top3   = pd.read_csv(shap_top3_path)
features    = pd.read_csv(features_path)

print(f"  Agent 2 predictions  : {len(predictions):,} applicants")
print(f"  SHAP Top3 records    : {len(shap_top3):,}")
print(f"  Agent 0 features     : {len(features):,} applicants")
print(f"  Columns in features  : {features.shape[1]}")

# ── Load Agent 3 graph scores (fallback to defaults if not built yet) ─────
if graph_path.exists():
    with open(graph_path) as f:
        graph_data = json.load(f)
    print(f"  Agent 3 graph scores : loaded ✅")
    AGENT3_AVAILABLE = True
else:
    print(f"  Agent 3 graph scores : not found — using behavioral proxy ⚠")
    AGENT3_AVAILABLE = False

print(f"\n  Predictions columns  : {list(predictions.columns)}")
print(f"  SHAP Top3 columns    : {list(shap_top3.columns)}")


## Step 2 — Social Trust Score

If Agent 3 is available, use the real graph scores.  
If not, compute a **behavioral proxy** from the features we already have.  
The proxy uses the same signals as the graph — recurring payees, obligation fulfillment, inflow stability.


In [ ]:
def compute_social_trust_proxy(row: pd.Series) -> float:
    """
    Behavioral proxy for Social Trust Score when Agent 3 is not available.
    
    Uses transaction-derived features that approximate graph metrics:
    - recurring_payment_ratio  → edge stability (same payees month after month)
    - obligation_fulfillment   → node reliability (always pays key counterparties)
    - inflow_outflow_ratio     → net flow stability
    - merchant_diversity       → network diversity
    - utility_streak           → temporal consistency
    
    Returns: float 0-100
    """
    eps = 1e-6

    # Edge stability proxy — recurring payees
    edge_stability = float(row.get('upi_recurring_payment_ratio', 0.5))

    # Node reliability — obligation fulfillment
    node_reliability = float(row.get('upi_obligation_fulfillment_rate', 0.5))

    # Flow stability — net saver signal
    ior = float(row.get('upi_inflow_outflow_ratio', 1.0))
    flow_stability = min(ior / 2.0, 1.0)  # normalise to 0-1, cap at 2.0

    # Network diversity — merchant spread
    diversity = float(row.get('upi_merchant_diversity_score', 0.5))

    # Temporal consistency — utility streak (normalise to 12 months)
    streak = float(row.get('upi_utility_payment_streak', 0))
    temporal = min(streak / 12.0, 1.0)

    # Weighted composite — same logic as graph metrics
    raw = (
        0.25 * edge_stability  +
        0.30 * node_reliability +
        0.20 * flow_stability  +
        0.15 * diversity       +
        0.10 * temporal
    )

    return round(float(np.clip(raw * 100, 0, 100)), 2)


def get_social_trust_score(applicant_token: str, 
                            features_row: pd.Series) -> dict:
    """Returns social trust score + component breakdown."""
    
    if AGENT3_AVAILABLE and applicant_token in graph_data:
        g = graph_data[applicant_token]
        score = g.get('social_trust_score', 50)
        return {
            'social_trust_score': score,
            'graph_centrality':   g.get('graph_centrality_score', None),
            'edge_stability':     g.get('edge_stability_index', None),
            'network_diversity':  g.get('network_diversity_score', None),
            'community_risk':     g.get('community_risk_flag', False),
            'payer_diversity':    g.get('payer_diversity_score', None),
            'source': 'agent3_graph',
        }
    else:
        score = compute_social_trust_proxy(features_row)
        return {
            'social_trust_score': score,
            'graph_centrality':   None,
            'edge_stability':     float(features_row.get('upi_recurring_payment_ratio', 0)),
            'network_diversity':  float(features_row.get('upi_merchant_diversity_score', 0)),
            'community_risk':     False,
            'payer_diversity':    int(features_row.get('upi_unique_merchants', 0)),
            'source': 'behavioral_proxy',
        }

print("✅ Social trust functions defined.")
print("   Using:", "Agent 3 graph" if AGENT3_AVAILABLE else "behavioral proxy")


## Step 3 — Final GigScore Formula (3 Layers)

### Layer 1: XGBoost Base Score
Sigmoid inversion of `default_prob`. Calibrated so that 35% default probability (3× the cohort average) maps to exactly 50/100 — because income irregularity inflates raw default probabilities for gig workers and the midpoint adjustment corrects for that.

### Layer 2: Social Trust Overlay (10% weight)
Payment network stability from Agent 3 (or behavioral proxy). Conservative weight — ML model stays dominant. Social trust can nudge ±3-4 points max, never overrides the model.

### Layer 3: Behavioral Bonus/Penalty
Rewards gig-worker-specific financial discipline signals. Max +5 points (all three signals must fire), max -8 points (severe stress indicators). Each threshold is empirically set from the cohort distribution.

### Income Stability Multiplier
PLFS benchmarking adjustment. If reported income is implausibly high vs government survey data → slight reduction. If income is realistic and stable → slight bonus.


In [ ]:
def compute_base_score(default_prob: float, 
                       k: float = 6, 
                       midpoint: float = 0.35) -> float:
    """
    Layer 1: Sigmoid inversion of XGBoost default probability.
    
    k=6       : steepness — difference between 0.15 and 0.20 prob ≈ 10 points
    midpoint  : 0.35 → 35% default prob maps to score of 50
                (corrects for gig worker income irregularity inflating raw probs)
    """
    return float(np.clip(100 / (1 + np.exp(k * (default_prob - midpoint))), 0, 100))


def compute_behavioral_adjustment(row: pd.Series) -> dict:
    """
    Layer 3: Behavioral bonus and penalty from gig-worker-specific signals.
    
    Bonus signals (max +5 total):
      +2  utility_streak >= 9 months  : deep financial discipline
      +2  inflow_outflow_ratio >= 1.20 : net saver despite irregular income
      +1  obligation_fulfillment >= 0.90: pays all fixed obligations

    Penalty signals (max -8 total):
      -3  inflow_outflow_ratio < 0.80  : spending more than earning
      -3  obligation_fulfillment < 0.50 : missing more than half obligations
      -2  utility_streak == 0           : no utility payment history
    """
    bonus   = 0
    penalty = 0
    reasons = []

    streak = float(row.get('upi_utility_payment_streak', 0))
    ior    = float(row.get('upi_inflow_outflow_ratio', 1.0))
    oblig  = float(row.get('upi_obligation_fulfillment_rate', 0.5))

    # Bonuses
    if streak >= 9:
        bonus += 2
        reasons.append({'type': 'bonus', 'points': +2,
                        'feature': 'upi_utility_payment_streak',
                        'reason': f'{int(streak)}-month utility payment streak'})
    if ior >= 1.20:
        bonus += 2
        reasons.append({'type': 'bonus', 'points': +2,
                        'feature': 'upi_inflow_outflow_ratio',
                        'reason': f'Net saver — earns {ior:.2f}x what is spent'})
    if oblig >= 0.90:
        bonus += 1
        reasons.append({'type': 'bonus', 'points': +1,
                        'feature': 'upi_obligation_fulfillment_rate',
                        'reason': f'Paid all obligations {oblig:.0%} of months'})

    # Penalties
    if ior < 0.80:
        penalty += 3
        reasons.append({'type': 'penalty', 'points': -3,
                        'feature': 'upi_inflow_outflow_ratio',
                        'reason': f'Spending more than earning (ratio={ior:.2f})'})
    if oblig < 0.50:
        penalty += 3
        reasons.append({'type': 'penalty', 'points': -3,
                        'feature': 'upi_obligation_fulfillment_rate',
                        'reason': f'Missing obligations {(1-oblig):.0%} of months'})
    if streak == 0:
        penalty += 2
        reasons.append({'type': 'penalty', 'points': -2,
                        'feature': 'upi_utility_payment_streak',
                        'reason': 'No utility payment history found'})

    return {
        'bonus':       bonus,
        'penalty':     penalty,
        'net':         bonus - penalty,
        'adjustments': reasons,
    }


def income_stability_multiplier(row: pd.Series) -> float:
    """
    PLFS benchmarking adjustment.
    
    income_vs_plfs_ratio available only if PLFS feature passed through Agent 1.
    Falls back to income_regularity_cv if not available.
    """
    # Try PLFS ratio first
    plfs_ratio = row.get('income_vs_plfs_ratio', None)
    if plfs_ratio is not None and not pd.isna(plfs_ratio):
        plfs_ratio = float(plfs_ratio)
        if plfs_ratio > 1.5:
            return 0.97    # income implausibly high vs survey data
        elif plfs_ratio < 0.30:
            return 0.98    # very low income — higher stress
        elif 0.50 < plfs_ratio < 1.2:
            cv = float(row.get('upi_income_regularity_cv', 0.35))
            if cv < 0.25:
                return 1.02  # realistic income AND stable
    
    # Fallback — use income regularity CV alone
    cv = float(row.get('upi_income_regularity_cv', 0.35))
    if cv < 0.15:
        return 1.01   # very stable income
    elif cv > 0.70:
        return 0.98   # very volatile income
    return 1.0


def calculate_final_gigscore(default_prob: float,
                              social_trust: dict,
                              features_row: pd.Series) -> dict:
    """
    Master GigScore formula — combines all 3 layers.
    
    Returns full breakdown dict for Agent 5 and Agent 6.
    """
    # Layer 1
    base = compute_base_score(default_prob)

    # Layer 2 — 10% social trust, 90% ML base
    st_score = float(social_trust.get('social_trust_score', 50))
    blended  = 0.90 * base + 0.10 * st_score

    # Layer 3
    adj = compute_behavioral_adjustment(features_row)
    after_adj = blended + adj['net']

    # Income multiplier
    multiplier = income_stability_multiplier(features_row)
    final = float(np.clip(after_adj * multiplier, 0, 100))

    return {
        'base_score':          round(base, 2),
        'social_trust_score':  round(st_score, 2),
        'blended_score':       round(blended, 2),
        'behavioral_net':      adj['net'],
        'behavioral_bonus':    adj['bonus'],
        'behavioral_penalty':  adj['penalty'],
        'behavioral_adjustments': adj['adjustments'],
        'income_multiplier':   round(multiplier, 4),
        'final_gigscore':      round(final, 1),
    }


def assign_tier(score: float) -> dict:
    """Maps GigScore to tier + lending parameters per master plan."""
    if score >= 80:
        return {
            'tier':           'Prime',
            'decision':       'APPROVED',
            'max_loan':       100000,
            'rate_min':       11.0,
            'rate_max':       13.0,
            'max_tenure_m':   36,
            'emoji':          '✅',
        }
    elif score >= 65:
        return {
            'tier':           'Near-Prime',
            'decision':       'APPROVED',
            'max_loan':       50000,
            'rate_min':       14.0,
            'rate_max':       17.0,
            'max_tenure_m':   24,
            'emoji':          '✅',
        }
    elif score >= 50:
        return {
            'tier':           'Sub-Prime',
            'decision':       'CONDITIONAL',
            'max_loan':       25000,
            'rate_min':       18.0,
            'rate_max':       22.0,
            'max_tenure_m':   12,
            'emoji':          '⚡',
        }
    else:
        return {
            'tier':           'High Risk',
            'decision':       'DECLINED',
            'max_loan':       0,
            'rate_min':       None,
            'rate_max':       None,
            'max_tenure_m':   None,
            'emoji':          '❌',
        }


print("✅ GigScore formula functions defined.")
print()
print("  Layer weights:")
print("    Layer 1 (XGBoost base)       : 90%")
print("    Layer 2 (Social Trust)       : 10%")
print("    Layer 3 (Behavioral adj)     : ±5 bonus / ±8 penalty")
print("    Income multiplier            : 0.97 – 1.02")


## Step 4 — SHAP-Based Risk Explanation

For each applicant, translate SHAP values into:
- **Positive signals** — features pulling the score UP (reducing default risk)
- **Risk factors** — features pulling the score DOWN (increasing default risk)

Plain English descriptions are keyed to feature names — no jargon in user-facing output.


In [ ]:
# Plain English descriptions for every feature
FEATURE_DESCRIPTIONS = {
    # UPI behavioral
    'upi_income_regularity_cv':       ('Income regularity',        'How consistent monthly income is'),
    'upi_utility_payment_streak':     ('Utility payment streak',   'Consecutive months of utility payments'),
    'upi_cash_crunch_recovery_speed': ('Cash recovery speed',      'How fast balance recovers after a dip'),
    'upi_spending_velocity':          ('Spending stability',       'How stable monthly spending is'),
    'upi_merchant_diversity_score':   ('Spending diversity',       'Variety of merchant categories'),
    'upi_inflow_outflow_ratio':       ('Savings ratio',            'Total earnings vs total spending'),
    'upi_recurring_payment_ratio':    ('Recurring payments',       'Fixed payments made consistently'),
    'upi_obligation_fulfillment_rate':('Obligation fulfillment',   'Fixed obligations paid on time'),
    'upi_fraud_rate':                 ('Fraud flag rate',          'Suspicious transaction frequency'),
    'upi_success_rate':               ('Payment success rate',     'Successful vs failed payments'),

    # Synthetic behavioral proxies
    'syn_income_regularity_cv':       ('Income consistency',       'Estimated income stability'),
    'syn_utility_streak':             ('Utility discipline',       'Estimated utility payment streak'),
    'syn_merchant_diversity':         ('Lifestyle stability',      'Estimated spending diversity'),
    'syn_inflow_outflow':             ('Net saving estimate',      'Estimated savings behavior'),
    'syn_spending_velocity':          ('Spending volatility',      'Estimated spending stability'),
    'syn_crunch_recovery':            ('Resilience estimate',      'Estimated recovery from cash crunches'),
    'syn_recurring_ratio':            ('Recurring behavior',       'Estimated recurring payment ratio'),
    'syn_obligation_rate':            ('Payment discipline',       'Estimated obligation fulfillment'),

    # Home Credit / proxy
    'ext_source_mean':                ('Bureau score avg',         'Average external credit score'),
    'ext_source_min':                 ('Bureau score min',         'Weakest external credit score'),
    'EXT_SOURCE_2':                   ('Bureau score',             'External credit bureau assessment'),
    'EXT_SOURCE_3':                   ('Bureau score',             'External credit bureau assessment'),
    'EXT_SOURCE_1':                   ('Bureau score',             'External credit bureau assessment'),
    'credit_income_ratio':            ('Loan-to-income ratio',     'Loan amount relative to annual income'),
    'annuity_income_ratio':           ('EMI-to-income ratio',      'Monthly EMI as fraction of income'),
    'years_employed':                 ('Employment duration',      'Years of consistent employment/gig work'),
    'employment_age_ratio':           ('Career stability',         'Employment tenure relative to age'),
    'income_instability_proxy':       ('Income volatility',        'Phone change frequency as instability signal'),
    'credit_seeking_intensity':       ('Credit hunger',            'Bureau enquiry frequency'),
    'stress_signal':                  ('Financial stress',         'Late applications + high credit ratio'),
    'lifestyle_stability':            ('Lifestyle stability',      'Family size vs geography indicator'),
    'repayment_stretch':              ('Repayment stretch',        'Annuity load adjusted for region'),
    'credit_utilization_pressure':    ('Credit pressure',          'Debt burden vs income and tenure'),
    'bureau_requests':                ('Credit enquiries',         'Number of credit bureau requests'),

    # Loan / credit features
    'loan_loan_income_ratio':         ('Peer loan-income ratio',   'Cohort average loan-to-income'),
    'loan_dti_ratio':                 ('Debt-to-income',           'Cohort debt-to-income ratio'),
    'loan_loan_default_rate':         ('Peer default rate',        'Default rate in similar income cohort'),
    'cred_debt_income_ratio':         ('Peer debt ratio',          'Cohort debt-to-income benchmark'),
    'cred_obligation_fulfillment':    ('Peer obligation rate',     'Cohort obligation fulfillment benchmark'),

    # Region
    'REGION_RATING_CLIENT':           ('Geographic risk',          'City tier credit risk rating'),
    'REGION_RATING_CLIENT_W_CITY':    ('City risk rating',         'City-level credit risk rating'),
    'age':                            ('Applicant age',            'Age of applicant'),
}


def get_feature_description(feature_name: str) -> tuple:
    """Returns (short_name, description) for a feature."""
    return FEATURE_DESCRIPTIONS.get(
        feature_name,
        (feature_name.replace('_', ' ').title(), f'Feature: {feature_name}')
    )


def build_explanation(applicant_id,
                      shap_top3_df: pd.DataFrame,
                      features_row: pd.Series,
                      gigscore_breakdown: dict) -> dict:
    """
    Builds structured explanation dict for one applicant.
    
    Combines:
    - SHAP Top3 from Agent 2 (data-driven feature importance)
    - Behavioral adjustments from Layer 3 (rule-based signals)
    - Feature values from agent0_features.csv
    """
    # Get SHAP rows for this applicant
    shap_row = shap_top3_df[shap_top3_df['applicant_id'] == applicant_id]

    positive_signals = []
    risk_factors     = []

    # ── SHAP-based signals ────────────────────────────────────────────────
    if len(shap_row) > 0:
        row = shap_row.iloc[0]
        for i in [1, 2, 3]:
            feat  = str(row.get(f'feature_{i}', ''))
            shval = float(row.get(f'shap_{i}', 0))
            fval  = features_row.get(feat, None)
            short, desc = get_feature_description(feat)

            entry = {
                'rank':          i,
                'feature':       feat,
                'short_name':    short,
                'description':   desc,
                'shap_value':    round(shval, 4),
                'feature_value': round(float(fval), 4) if fval is not None and not pd.isna(fval) else None,
                'source':        'shap',
            }

            # Negative SHAP = reduces default risk = positive for applicant
            if shval < 0:
                positive_signals.append(entry)
            else:
                risk_factors.append(entry)

    # ── Layer 3 behavioral adjustments ───────────────────────────────────
    for adj in gigscore_breakdown.get('behavioral_adjustments', []):
        feat  = adj['feature']
        fval  = features_row.get(feat, None)
        short, desc = get_feature_description(feat)

        entry = {
            'feature':       feat,
            'short_name':    short,
            'description':   desc,
            'points':        adj['points'],
            'reason':        adj['reason'],
            'feature_value': round(float(fval), 4) if fval is not None and not pd.isna(fval) else None,
            'source':        'behavioral_rule',
        }

        if adj['type'] == 'bonus':
            # Add if not already captured by SHAP
            if not any(s['feature'] == feat for s in positive_signals):
                positive_signals.append(entry)
        else:
            if not any(r['feature'] == feat for r in risk_factors):
                risk_factors.append(entry)

    # ── Thin file flag ────────────────────────────────────────────────────
    ext_missing = pd.isna(features_row.get('EXT_SOURCE_2', None))
    if ext_missing:
        risk_factors.append({
            'feature':       'EXT_SOURCE_2',
            'short_name':    'No bureau history',
            'description':   'No formal credit history — scored on behavior only',
            'shap_value':    None,
            'feature_value': None,
            'source':        'thin_file_flag',
        })

    # ── Confidence interval ───────────────────────────────────────────────
    default_prob = float(features_row.get('default_prob',
                         1 - gigscore_breakdown['final_gigscore'] / 100))
    ci_low  = round(max(default_prob - 0.035, 0), 3)
    ci_high = round(min(default_prob + 0.035, 1), 3)

    return {
        'positive_signals': positive_signals[:3],   # top 3 only
        'risk_factors':     risk_factors[:3],        # top 3 only
        'confidence_interval': [ci_low, ci_high],
        'confidence_note':  f"Model is 90% confident default probability is between {ci_low:.0%}–{ci_high:.0%}",
        'thin_file':        bool(ext_missing),
        'explanation_complete': True,
    }


print("✅ Explanation functions defined.")
print(f"   Feature descriptions available: {len(FEATURE_DESCRIPTIONS)}")


## Step 5 — Structured Reasoning Generator

Produces a human-readable reasoning string for Agent 5 (credit story) and the Barclays dashboard.  
Logic trees based on the actual score drivers — not a template.


In [ ]:
def generate_reasoning(features_row: pd.Series,
                       gigscore_breakdown: dict,
                       explanation: dict,
                       tier_info: dict) -> str:
    """
    Generates structured reasoning string for Agent 5.
    
    Covers: income pattern, payment discipline, savings behavior,
            credit history, overall assessment.
    """
    final   = gigscore_breakdown['final_gigscore']
    tier    = tier_info['tier']
    cv      = float(features_row.get('upi_income_regularity_cv', 0.35))
    streak  = float(features_row.get('upi_utility_payment_streak', 0))
    ior     = float(features_row.get('upi_inflow_outflow_ratio', 1.0))
    oblig   = float(features_row.get('upi_obligation_fulfillment_rate', 0.5))
    div     = float(features_row.get('upi_merchant_diversity_score', 0.5))
    thin    = explanation['thin_file']
    bonus   = gigscore_breakdown['behavioral_bonus']
    penalty = gigscore_breakdown['behavioral_penalty']

    parts = []

    # Income pattern
    if cv < 0.20:
        parts.append(f"Income is highly consistent (CV={cv:.2f}) — unusually stable for a gig worker.")
    elif cv < 0.35:
        parts.append(f"Income shows normal gig worker variability (CV={cv:.2f}) — not a risk signal.")
    else:
        parts.append(f"Income is irregular (CV={cv:.2f}) — elevated variability warrants monitoring.")

    # Payment discipline
    if streak >= 12:
        parts.append(f"Perfect utility payment streak of {int(streak)} months — strongest creditworthiness signal.")
    elif streak >= 6:
        parts.append(f"Strong utility payment streak of {int(streak)} months — demonstrates financial discipline.")
    elif streak >= 3:
        parts.append(f"Partial utility payment streak ({int(streak)} months) — some discipline shown.")
    else:
        parts.append(f"Weak utility payment history ({int(streak)} months) — key risk signal.")

    # Savings behavior
    if ior >= 1.5:
        parts.append(f"Strong net saver — earns {ior:.2f}x what is spent. Financial buffer present.")
    elif ior >= 1.2:
        parts.append(f"Net saver (ratio={ior:.2f}) — consistently spending less than earning.")
    elif ior >= 0.9:
        parts.append(f"Near break-even spending (ratio={ior:.2f}) — limited savings buffer.")
    else:
        parts.append(f"Spending exceeds income (ratio={ior:.2f}) — financial stress indicator.")

    # Obligation fulfillment
    if oblig >= 0.90:
        parts.append(f"Pays all fixed obligations {oblig:.0%} of months — core creditworthiness confirmed.")
    elif oblig >= 0.70:
        parts.append(f"Meets obligations {oblig:.0%} of months — generally reliable.")
    else:
        parts.append(f"Misses obligations {(1-oblig):.0%} of months — repayment risk elevated.")

    # Merchant diversity
    if div >= 0.75:
        parts.append(f"High merchant diversity ({div:.2f}) — stable lifestyle across multiple spending categories.")
    elif div >= 0.50:
        parts.append(f"Moderate spending diversity ({div:.2f}) — some lifestyle stability.")
    else:
        parts.append(f"Low spending diversity ({div:.2f}) — concentrated or survival-mode spending pattern.")

    # Bureau history
    if thin:
        parts.append("No formal credit bureau history — decision based entirely on behavioral signals.")
    else:
        ext = float(features_row.get('ext_source_mean', 0.45))
        parts.append(f"External credit score present (mean={ext:.2f}) — bureau signal incorporated.")

    # Behavioral adjustment summary
    if bonus > 0 and penalty == 0:
        parts.append(f"Behavioral layer added +{bonus} points — all positive signals aligned.")
    elif bonus > 0 and penalty > 0:
        parts.append(f"Behavioral layer: +{bonus} bonus offset by -{penalty} penalty = net {bonus-penalty:+d} points.")
    elif penalty > 0:
        parts.append(f"Behavioral layer applied -{penalty} point penalty — stress signals detected.")

    # Final assessment
    if tier == 'Prime':
        parts.append(f"Overall: Exceptional financial discipline. GigScore {final} — Prime tier. Recommended for approval.")
    elif tier == 'Near-Prime':
        parts.append(f"Overall: Strong behavioral signals despite thin file. GigScore {final} — Near-Prime. Recommended for approval with standard terms.")
    elif tier == 'Sub-Prime':
        parts.append(f"Overall: Mixed signals. GigScore {final} — Sub-Prime. Conditional approval with reduced limit and higher rate.")
    else:
        parts.append(f"Overall: Insufficient behavioral evidence for approval. GigScore {final} — Declined. 90-day improvement plan generated.")

    return ' '.join(parts)


print("✅ Reasoning generator defined.")


## Step 6 — 90-Day Improvement Plan (Declined Applicants)

When GigScore declines an applicant, Agent 4 generates a specific, actionable 90-day plan.  
**No bank in India does this. GigScore turns a rejection into a roadmap.**


In [ ]:
def generate_improvement_plan(features_row: pd.Series,
                               final_gigscore: float) -> dict:
    """
    Generates actionable 90-day improvement plan for declined applicants.
    Each action targets a specific behavioral feature with a projected score impact.
    """
    actions      = []
    projected_lift = 0

    streak = float(features_row.get('upi_utility_payment_streak', 0))
    ior    = float(features_row.get('upi_inflow_outflow_ratio', 1.0))
    oblig  = float(features_row.get('upi_obligation_fulfillment_rate', 0.5))
    div    = float(features_row.get('upi_merchant_diversity_score', 0.5))
    cv     = float(features_row.get('upi_income_regularity_cv', 0.35))

    # Action 1 — Utility payment streak
    if streak < 6:
        impact = 8 if streak == 0 else 6
        actions.append({
            'priority':        1,
            'action':          'Pay electricity, water and phone bills before the 5th of each month',
            'current_behavior': f'Current streak: {int(streak)} months',
            'target':          'Maintain streak for 3 consecutive months',
            'score_impact':    f'+{impact} points',
            'impact_points':   impact,
            'feature':         'upi_utility_payment_streak',
            'why':             'Utility payment streak is the strongest creditworthiness signal for gig workers',
        })
        projected_lift += impact

    # Action 2 — Reduce cash withdrawals
    if ior < 1.0:
        actions.append({
            'priority':        2,
            'action':          'Reduce ATM cash withdrawals — use UPI for all payments above ₹200',
            'current_behavior': f'Current ratio: {ior:.2f} (spending more than earning)',
            'target':          'Achieve inflow/outflow ratio above 1.2',
            'score_impact':    '+6 points',
            'impact_points':   6,
            'feature':         'upi_inflow_outflow_ratio',
            'why':             'Net saving behavior — even small surplus — is a strong repayment signal',
        })
        projected_lift += 6

    # Action 3 — Merchant diversity
    if div < 0.50:
        actions.append({
            'priority':        3,
            'action':          'Increase UPI merchant payments to 15+ different categories per month',
            'current_behavior': f'Current diversity score: {div:.2f}',
            'target':          'Achieve diversity score above 0.60',
            'score_impact':    '+5 points',
            'impact_points':   5,
            'feature':         'upi_merchant_diversity_score',
            'why':             'Diverse spending patterns indicate stable lifestyle and financial planning',
        })
        projected_lift += 5

    # Action 4 — Obligation fulfillment
    if oblig < 0.70:
        impact = 7
        actions.append({
            'priority':        4,
            'action':          'Pay rent and all utilities every month without fail for 3 months',
            'current_behavior': f'Currently fulfilling obligations {oblig:.0%} of months',
            'target':          'Achieve 100% obligation fulfillment for 3 consecutive months',
            'score_impact':    f'+{impact} points',
            'impact_points':   impact,
            'feature':         'upi_obligation_fulfillment_rate',
            'why':             'Obligation fulfillment rate directly predicts loan repayment behavior',
        })
        projected_lift += impact

    # Action 5 — Income consistency
    if cv > 0.50:
        actions.append({
            'priority':        5,
            'action':          'Accept platform orders consistently — aim for 25+ working days per month',
            'current_behavior': f'Income variability (CV): {cv:.2f} — high volatility',
            'target':          'Reduce income CV below 0.35',
            'score_impact':    '+4 points',
            'impact_points':   4,
            'feature':         'upi_income_regularity_cv',
            'why':             'Consistent gig platform income signals reliability to the model',
        })
        projected_lift += 4

    projected_score = round(min(final_gigscore + projected_lift, 100), 1)

    # Determine projected tier
    projected_tier_info = assign_tier(projected_score)

    return {
        'current_score':    final_gigscore,
        'projected_score':  projected_score,
        'projected_lift':   projected_lift,
        'projected_tier':   projected_tier_info['tier'],
        'projected_decision': projected_tier_info['decision'],
        'projected_max_loan': projected_tier_info['max_loan'],
        'actions':          actions[:4],   # top 4 actions
        'timeframe':        '90 days',
        'message':          f"Follow this plan for 90 days to reach GigScore {projected_score} — {projected_tier_info['emoji']} {projected_tier_info['decision']} ₹{projected_tier_info['max_loan']:,}",
    }


print("✅ 90-day improvement plan generator defined.")


## Step 7 — Run Agent 4 Pipeline

Process every applicant through the full explanation + scoring pipeline.


In [ ]:
def run_agent4_for_applicant(applicant_token: str,
                              features_row: pd.Series,
                              predictions_df: pd.DataFrame,
                              shap_top3_df: pd.DataFrame) -> dict:
    """
    Full Agent 4 pipeline for one applicant.
    
    Returns complete output dict ready for Agent 5 and Agent 6.
    """
    # ── Get Agent 2 outputs ───────────────────────────────────────────────
    pred_row = predictions_df[predictions_df['applicant_id'] == applicant_token]

    if len(pred_row) > 0:
        default_prob  = float(pred_row.iloc[0]['default_prob'])
        raw_gigscore  = float(pred_row.iloc[0]['gigscore'])
        actual_default = pred_row.iloc[0].get('actual_default', None)
    else:
        # Fallback — compute from features if predictions not found
        print(f"  ⚠ No prediction found for {applicant_token} — using feature-based estimate")
        default_prob  = 0.15
        raw_gigscore  = compute_base_score(default_prob)
        actual_default = None

    # ── Get Agent 3 social trust ──────────────────────────────────────────
    social_trust = get_social_trust_score(applicant_token, features_row)

    # ── Layer 1+2+3: Final GigScore ───────────────────────────────────────
    gigscore_breakdown = calculate_final_gigscore(
        default_prob, social_trust, features_row
    )

    # ── Tier assignment ───────────────────────────────────────────────────
    final_score = gigscore_breakdown['final_gigscore']
    tier_info   = assign_tier(final_score)

    # ── SHAP explanation ──────────────────────────────────────────────────
    # Add default_prob to features_row for confidence interval calc
    features_row_copy = features_row.copy()
    features_row_copy['default_prob'] = default_prob

    explanation = build_explanation(
        applicant_token, shap_top3_df, features_row_copy, gigscore_breakdown
    )

    # ── Reasoning string ──────────────────────────────────────────────────
    reasoning = generate_reasoning(
        features_row, gigscore_breakdown, explanation, tier_info
    )

    # ── 90-day improvement plan (declined only) ───────────────────────────
    improvement_plan = None
    if tier_info['decision'] == 'DECLINED':
        improvement_plan = generate_improvement_plan(features_row, final_score)

    # ── Assemble output ───────────────────────────────────────────────────
    return {
        # Identity
        'applicant_token':   applicant_token,
        'persona_id':        features_row.get('persona_id', applicant_token),

        # Final score
        'final_gigscore':    final_score,
        'raw_gigscore':      raw_gigscore,
        'default_prob':      default_prob,
        'tier':              tier_info['tier'],
        'decision':          tier_info['decision'],
        'emoji':             tier_info['emoji'],

        # Score breakdown
        'score_breakdown':   gigscore_breakdown,
        'tier_info':         tier_info,

        # Social trust
        'social_trust':      social_trust,

        # Explanation
        'positive_signals':  explanation['positive_signals'],
        'risk_factors':      explanation['risk_factors'],
        'confidence_interval': explanation['confidence_interval'],
        'confidence_note':   explanation['confidence_note'],
        'thin_file':         explanation['thin_file'],

        # Reasoning
        'reasoning':         reasoning,

        # Improvement plan
        'improvement_plan':  improvement_plan,

        # Pass-through for Agent 6
        'monthly_income':    float(features_row.get('AMT_INCOME_MONTHLY',
                             features_row.get('AMT_INCOME_TOTAL', 240000) / 12)),
        'loan_amount':       float(features_row.get('AMT_CREDIT', 50000)),

        # Actual outcome (training data only, None for live inference)
        'actual_default':    actual_default,
    }


# ── Run for all applicants ────────────────────────────────────────────────
print("Running Agent 4 pipeline...")
print("=" * 68)

agent4_outputs = {}

for idx, feat_row in features.iterrows():
    token    = str(feat_row.get('applicant_token', f'APPL_{idx}'))
    persona  = str(feat_row.get('persona_id', f'applicant_{idx}'))

    output = run_agent4_for_applicant(
        applicant_token = token,
        features_row    = feat_row,
        predictions_df  = predictions,
        shap_top3_df    = shap_top3,
    )

    agent4_outputs[token] = output

    print(f"\n  {persona.replace('_', ' ').title()}")
    print(f"  {'─' * 50}")
    print(f"  Raw GigScore (Agent 2)  : {output['raw_gigscore']:.1f}")
    print(f"  Social Trust            : {output['social_trust']['social_trust_score']:.1f}  [{output['social_trust']['source']}]")
    print(f"  Behavioral adjustment   : {output['score_breakdown']['behavioral_net']:+d}  (bonus={output['score_breakdown']['behavioral_bonus']}, penalty={output['score_breakdown']['behavioral_penalty']})")
    print(f"  Income multiplier       : {output['score_breakdown']['income_multiplier']:.4f}")
    print(f"  ──────────────────────────────────")
    print(f"  FINAL GigScore          : {output['final_gigscore']} / 100")
    print(f"  Tier                    : {output['emoji']} {output['tier']}")
    print(f"  Decision                : {output['decision']}")
    print(f"  Confidence interval     : {output['confidence_interval']}")
    print(f"  Thin file               : {output['thin_file']}")

    if output['positive_signals']:
        print(f"\n  Top positive signals:")
        for s in output['positive_signals']:
            val = f"= {s['feature_value']}" if s.get('feature_value') is not None else ""
            print(f"    ✅ {s['short_name']} {val}")

    if output['risk_factors']:
        print(f"\n  Top risk factors:")
        for r in output['risk_factors']:
            val = f"= {r['feature_value']}" if r.get('feature_value') is not None else ""
            print(f"    ⚠  {r['short_name']} {val}")

    if output['improvement_plan']:
        plan = output['improvement_plan']
        print(f"\n  90-Day Plan: {plan['message']}")
        for a in plan['actions'][:2]:
            print(f"    → {a['action']}  [{a['score_impact']}]")

print(f"\n{'=' * 68}")
print(f"Agent 4 complete. {len(agent4_outputs)} applicants processed.")


## Step 8 — Visualisations

In [ ]:
if not agent4_outputs:
    print("No outputs to visualise.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle("GigScore Agent 4 — Risk Explanation Dashboard",
                 fontsize=15, fontweight='bold')

    tokens   = list(agent4_outputs.keys())
    personas = [agent4_outputs[t]['persona_id'].replace('_', ' ').title() for t in tokens]
    finals   = [agent4_outputs[t]['final_gigscore']  for t in tokens]
    raws     = [agent4_outputs[t]['raw_gigscore']     for t in tokens]
    sts      = [agent4_outputs[t]['social_trust']['social_trust_score'] for t in tokens]
    bonuses  = [agent4_outputs[t]['score_breakdown']['behavioral_bonus']   for t in tokens]
    penalties= [agent4_outputs[t]['score_breakdown']['behavioral_penalty'] for t in tokens]

    tier_colours = {
        'Prime':      '#166534',
        'Near-Prime': BARCLAYS_NAVY,
        'Sub-Prime':  BARCLAYS_EAGLE,
        'High Risk':  '#9CA3AF',
    }
    bar_colours = [tier_colours[agent4_outputs[t]['tier']] for t in tokens]

    # ── Chart 1: Score Breakdown ──────────────────────────────────────────
    ax = axes[0, 0]
    x  = np.arange(len(tokens))
    w  = 0.25
    ax.bar(x - w, raws,   width=w, label='XGBoost Base',   color=BARCLAYS_NAVY,  alpha=0.8)
    ax.bar(x,     sts,    width=w, label='Social Trust',   color=BARCLAYS_BLUE,  alpha=0.8)
    ax.bar(x + w, finals, width=w, label='Final GigScore', color=BARCLAYS_EAGLE, alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(personas, rotation=15, ha='right')
    ax.set_ylabel('Score (0–100)')
    ax.set_title('Score Comparison: Base vs Social Trust vs Final', fontweight='bold')
    ax.legend(fontsize=9); ax.set_ylim(0, 110)
    for i, (r, s, f) in enumerate(zip(raws, sts, finals)):
        ax.text(i - w, r + 1, f'{r:.0f}', ha='center', fontsize=8)
        ax.text(i,     s + 1, f'{s:.0f}', ha='center', fontsize=8)
        ax.text(i + w, f + 1, f'{f:.0f}', ha='center', fontsize=8, fontweight='bold')

    # ── Chart 2: Final GigScore by Tier ──────────────────────────────────
    ax = axes[0, 1]
    bars = ax.bar(personas, finals, color=bar_colours, edgecolor='white', linewidth=0.5)
    ax.axhline(80, color='#166534',    linestyle='--', linewidth=1.5, label='Prime (80+)')
    ax.axhline(65, color=BARCLAYS_NAVY,linestyle='--', linewidth=1.5, label='Near-Prime (65+)')
    ax.axhline(50, color=BARCLAYS_EAGLE,linestyle='--', linewidth=1.5, label='Sub-Prime (50+)')
    for bar, val, tok in zip(bars, finals, tokens):
        tier = agent4_outputs[tok]['tier']
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.0f}\n{tier}", ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel('Final GigScore')
    ax.set_title('Final GigScore by Applicant', fontweight='bold')
    ax.legend(fontsize=9); ax.set_ylim(0, 115)
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right')

    # ── Chart 3: Behavioral Adjustment ───────────────────────────────────
    ax = axes[1, 0]
    x  = np.arange(len(tokens))
    ax.bar(x - 0.2, bonuses,   width=0.35, label='Bonus',   color=GREEN,          alpha=0.8)
    ax.bar(x + 0.2, [-p for p in penalties], width=0.35, label='Penalty', color=BARCLAYS_EAGLE, alpha=0.8)
    nets = [b - p for b, p in zip(bonuses, penalties)]
    ax.plot(x, nets, 'ko--', linewidth=2, markersize=6, label='Net adjustment', zorder=5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels(personas, rotation=15, ha='right')
    ax.set_ylabel('Points')
    ax.set_title('Layer 3: Behavioral Bonus / Penalty', fontweight='bold')
    ax.legend(fontsize=9)

    # ── Chart 4: Score Waterfall for first applicant ──────────────────────
    ax = axes[1, 1]
    tok   = tokens[0]
    out   = agent4_outputs[tok]
    bd    = out['score_breakdown']
    steps = ['XGBoost\nBase', '+Social\nTrust', '+Behavioral\nAdj', '×Income\nMultiplier', 'Final\nGigScore']
    vals  = [
        bd['base_score'],
        bd['blended_score'] - bd['base_score'],
        bd['behavioral_net'],
        (bd['blended_score'] + bd['behavioral_net']) * (bd['income_multiplier'] - 1),
        bd['final_gigscore'],
    ]
    running = 0
    bottoms = []
    colours_w = []
    for i, (step, val) in enumerate(zip(steps, vals)):
        if i == len(steps) - 1:
            bottoms.append(0)
            colours_w.append(tier_colours[out['tier']])
        else:
            bottoms.append(running)
            colours_w.append(GREEN if val >= 0 else BARCLAYS_EAGLE)
            running += val

    bars_w = ax.bar(steps, [abs(v) for v in vals],
                    bottom=bottoms, color=colours_w, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars_w, vals):
        ypos = bar.get_y() + bar.get_height() / 2
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{val:+.1f}' if abs(val) > 0.01 else '',
                ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.set_ylabel('Score')
    ax.set_title(f'Score Waterfall — {out["persona_id"].replace("_"," ").title()}\nFinal: {out["final_gigscore"]}/100 ({out["tier"]})',
                 fontweight='bold')
    ax.set_ylim(0, 115)

    plt.tight_layout()
    plt.savefig('chart_agent4_dashboard.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("  Agent 4 dashboard saved.")


## Step 9 — Save Outputs for Agent 5 + Agent 6

In [ ]:
import json

# ── Save full output as JSON for Agent 5 and Agent 6 ─────────────────────
output_json_path = 'agent4_output.json'
with open(output_json_path, 'w') as f:
    json.dump(agent4_outputs, f, indent=2, default=str)

# ── Save summary CSV ──────────────────────────────────────────────────────
summary_rows = []
for token, out in agent4_outputs.items():
    summary_rows.append({
        'applicant_token':    token,
        'persona_id':         out.get('persona_id', token),
        'raw_gigscore':       out['raw_gigscore'],
        'social_trust_score': out['social_trust']['social_trust_score'],
        'behavioral_bonus':   out['score_breakdown']['behavioral_bonus'],
        'behavioral_penalty': out['score_breakdown']['behavioral_penalty'],
        'income_multiplier':  out['score_breakdown']['income_multiplier'],
        'final_gigscore':     out['final_gigscore'],
        'tier':               out['tier'],
        'decision':           out['decision'],
        'default_prob':       out['default_prob'],
        'thin_file':          out['thin_file'],
        'ci_low':             out['confidence_interval'][0],
        'ci_high':            out['confidence_interval'][1],
        'monthly_income':     out['monthly_income'],
        'loan_amount':        out['loan_amount'],
        'has_improvement_plan': out['improvement_plan'] is not None,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('agent4_summary.csv', index=False)

print("╔══════════════════════════════════════════════════════════════════╗")
print("║                    AGENT 4 COMPLETE                             ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Applicants processed : {len(agent4_outputs):>3}                                    ║")
print("╠══════════════════════════════════════════════════════════════════╣")

for token, out in agent4_outputs.items():
    persona = out.get('persona_id', token).replace('_', ' ').title()
    print(f"║  {persona:<20}  GigScore: {out['final_gigscore']:>5.1f}  {out['emoji']} {out['tier']:<12}    ║")

print("╠══════════════════════════════════════════════════════════════════╣")
print("║  FILES SAVED                                                     ║")
print("║  📄 agent4_output.json        → Agent 5 (credit story)          ║")
print("║  📄 agent4_summary.csv        → Agent 6 (loan structuring)      ║")
print("║  📊 chart_agent4_dashboard.png                                   ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print("║  NEXT: Agent 5 — Credit Story (User + Barclays)                 ║")
print("║        Agent 6 — Loan Structuring + 90-Day Plan                 ║")
print("╚══════════════════════════════════════════════════════════════════╝")
